In [ ]:
###
# 1. 環境のセットアップ
###

import sys
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

ROOT_PATH = Path('/content/drive/MyDrive/cnn-hands-on')
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

os.chdir(ROOT_PATH)

# 日本語フォント対応
!pip install -q japanize-matplotlib
import japanize_matplotlib

print(f"✅ 環境セットアップ完了！現在のディレクトリ: {Path.cwd()}")

In [ ]:
###
# 2. 必要なライブラリのインポート
###

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# GPUが使える場合はGPUを、使えない場合はCPUを使用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用するデバイス: {device}")

# 3. MNISTデータの読み込み

MNISTは手書き数字（0〜9）の画像データセット。
- 訓練データ: 60,000枚
- テストデータ: 10,000枚
- 画像サイズ: 28×28 グレースケール

In [ ]:
###
# 3-1. データの前処理とダウンロード
###

# 前処理: 画像をテンソルに変換
transform = transforms.Compose([
    transforms.ToTensor(),
])

# MNISTデータセットのダウンロード
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform,
)

print(f"訓練データ: {len(train_dataset)}枚")
print(f"テストデータ: {len(test_dataset)}枚")

In [ ]:
###
# 3-2. DataLoaderの作成
###

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
)

# データの形状を確認
images, labels = next(iter(train_loader))
print(f"画像バッチの形状: {images.shape}")  # (64, 1, 28, 28)
print(f"ラベルバッチの形状: {labels.shape}")  # (64,)
print(f"ピクセル値の範囲: {images.min():.2f} 〜 {images.max():.2f}")

In [ ]:
###
# 3-3. サンプル画像の表示
###

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f"ラベル: {labels[i].item()}", fontsize=12)
    ax.axis('off')

plt.suptitle('MNISTサンプル画像', fontsize=14)
plt.tight_layout()
plt.show()

---

# 4. 損失関数を体験する

In [ ]:
###
# 4-1. CrossEntropyLossの動作確認
###

criterion = nn.CrossEntropyLoss()

# ケース1: 正解が「0」で、モデルが「0」を高スコアで予測
good_output = torch.tensor([[5.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]])
label = torch.tensor([0])
loss_good = criterion(good_output, label)
print(f"良い予測の損失: {loss_good.item():.4f}")

# ケース2: 正解が「0」で、モデルが「5」を高スコアで予測
bad_output = torch.tensor([[0.1, 0.1, 0.1, 0.1, 0.1, 5.0, 0.1, 0.1, 0.1, 0.1]])
loss_bad = criterion(bad_output, label)
print(f"悪い予測の損失: {loss_bad.item():.4f}")

# ケース3: ランダムな予測
random_output = torch.randn(1, 10)
loss_random = criterion(random_output, label)
print(f"ランダム予測の損失: {loss_random.item():.4f}")

print(f"\n→ 良い予測ほど損失が小さい！")

---

# 5. CNNモデルの定義

第4回で学んだネットワーク構築の知識を使って、MNIST用のCNNを定義する

In [ ]:
###
# 5. CNNモデルの定義
###

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 畳み込みブロック1
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        # 畳み込みブロック2
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # プーリング
        self.pool = nn.MaxPool2d(2)
        # 全結合層
        # 28 -> 14 (pool1) -> 7 (pool2)
        # 32チャンネル × 7 × 7 = 1568
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)  # 10クラス

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # (1,28,28) -> (16,14,14)
        x = self.pool(F.relu(self.conv2(x)))  # (16,14,14) -> (32,7,7)
        x = torch.flatten(x, 1)                # (32,7,7) -> (1568,)
        x = F.relu(self.fc1(x))                 # (1568,) -> (128,)
        x = self.fc2(x)                         # (128,) -> (10,)
        return x

model = SimpleCNN().to(device)
print(model)

# パラメータ数の確認
total_params = sum(p.numel() for p in model.parameters())
print(f"\n総パラメータ数: {total_params:,}")

---

# 6. 学習ループの実装

損失関数とオプティマイザを設定し、学習ループを回す

In [ ]:
###
# 6-1. 損失関数とオプティマイザの設定
###

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"損失関数: {criterion}")
print(f"オプティマイザ: Adam (lr=0.001)")

In [ ]:
###
# 6-2. 学習ループ
###

num_epochs = 5
train_loss_list = []
test_loss_list = []
test_acc_list = []

print("学習開始!")
for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # 学習の4ステップ
        optimizer.zero_grad()               # ① 勾配リセット
        outputs = model(images)             # ② 順伝播
        loss = criterion(outputs, labels)   # ③ 損失計算
        loss.backward()                     # ④ 逆伝播
        optimizer.step()                    # ⑤ パラメータ更新

        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)
    train_loss_list.append(epoch_train_loss)

    # --- Test (Validation) ---
    model.eval()
    running_test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_test_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_test_loss = running_test_loss / len(test_loader)
    epoch_test_acc = 100 * correct / total

    test_loss_list.append(epoch_test_loss)
    test_acc_list.append(epoch_test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {epoch_train_loss:.4f} | "
          f"Test Loss: {epoch_test_loss:.4f} | "
          f"Test Acc: {epoch_test_acc:.2f}%")

print("\n学習完了!")

---

# 7. 学習結果の可視化

In [ ]:
###
# 7. 学習曲線の描画
###

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# --- Loss ---
ax1.plot(range(1, num_epochs+1), train_loss_list, label='Train Loss', marker='o', color='blue')
ax1.plot(range(1, num_epochs+1), test_loss_list, label='Test Loss', marker='o', color='orange')
ax1.set_title('Loss（損失）の推移')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# --- Accuracy ---
ax2.plot(range(1, num_epochs+1), test_acc_list, label='Test Accuracy', marker='o', color='green')
ax2.set_title('Accuracy（正答率）の推移')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

---

# 8. 推論: テスト画像で予測してみよう

In [ ]:
###
# 8. テスト画像での推論
###

model.eval()

# テストデータからランダムに10枚取得
test_images, test_labels = next(iter(test_loader))
test_images = test_images[:10].to(device)
test_labels = test_labels[:10]

with torch.no_grad():
    outputs = model(test_images)
    _, predicted = torch.max(outputs, 1)

# 結果を表示
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i].cpu().squeeze(), cmap='gray')
    pred = predicted[i].item()
    true = test_labels[i].item()
    color = 'green' if pred == true else 'red'
    ax.set_title(f"予測: {pred} (正解: {true})", fontsize=11, color=color)
    ax.axis('off')

plt.suptitle('推論結果（緑=正解、赤=不正解）', fontsize=14)
plt.tight_layout()
plt.show()